# Setup

### Loading Data

In [1]:
%pip install -q kagglehub

from pathlib import Path
import pandas as pd
import kagglehub

dataset_path = Path(
    kagglehub.dataset_download("olistbr/brazilian-ecommerce")
)

orders_file = next(dataset_path.rglob("olist_orders_dataset.csv"))
RAW_DIR = orders_file.parent

csv_files = sorted(RAW_DIR.glob("*.csv"))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


### Loading Tables

In [2]:
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")



---


# Identifier and Categorical Cleaning

### Check categorical values

In [3]:
print("Order statuses:")
print(orders["order_status"].value_counts(dropna=False))

print("\nPayment types:")
print(payments["payment_type"].value_counts(dropna=False))

print("\nCustomer states:")
print(customers["customer_state"].value_counts(dropna=False).sort_index())

print("\nSeller states:")
print(sellers["seller_state"].value_counts(dropna=False).sort_index())

Order statuses:
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Payment types:
payment_type
credit_card    76795
boleto         19784
voucher         5775
debit_card      1529
not_defined        3
Name: count, dtype: int64

Customer states:
customer_state
AC       81
AL      413
AM      148
AP       68
BA     3380
CE     1336
DF     2140
ES     2033
GO     2020
MA      747
MG    11635
MS      715
MT      907
PA      975
PB      536
PE     1652
PI      495
PR     5045
RJ    12852
RN      485
RO      253
RR       46
RS     5466
SC     3637
SE      350
SP    41746
TO      280
Name: count, dtype: int64

Seller states:
seller_state
AC       1
AM       1
BA      19
CE      13
DF      30
ES      23
GO      40
MA       1
MG     244
MS       5
MT       4
PA       1
PB       6
PE       9
PI       1
PR     349
RJ     171
RN       5
RO     

### Inspect city formatting

In [5]:
print("Customer city examples:")
print(sorted(customers["customer_city"].dropna().unique())[:50])

print("\nSeller city examples:")
print(sorted(sellers["seller_city"].dropna().unique())[:50])

Customer city examples:
['abadia dos dourados', 'abadiania', 'abaete', 'abaetetuba', 'abaiara', 'abaira', 'abare', 'abatia', 'abdon batista', 'abelardo luz', 'abrantes', 'abre campo', 'abreu e lima', 'acaiaca', 'acailandia', 'acajutiba', 'acarau', 'acari', 'acegua', 'acopiara', 'acreuna', 'acu', 'acucena', 'adamantina', 'adhemar de barros', 'adolfo', 'adrianopolis', 'adustina', 'afogados da ingazeira', 'afonso claudio', 'afranio', 'agisse', 'agrestina', 'agrolandia', 'agronomica', 'agua boa', 'agua branca', 'agua clara', 'agua comprida', 'agua doce', 'agua doce do norte', 'agua fria de goias', 'agua limpa', 'agua nova', 'agua preta', 'agua santa', 'aguai', 'aguas belas', 'aguas claras', 'aguas da prata']

Seller city examples:
['04482255', 'abadia de goias', 'afonso claudio', 'aguas claras df', 'alambari', 'alfenas', 'almirante tamandare', 'alvares machado', 'alvorada', 'americana', 'amparo', 'ampere', 'anapolis', 'andira-pr', 'andradas', 'angra dos reis', 'angra dos reis rj', 'ao bern

### Inspect product categories

In [6]:
print("Unique product categories:")
print(products["product_category_name"].nunique(dropna=True))

print(
    products["product_category_name"]
    .value_counts(dropna=False)
    .head(20)
)

Unique product categories:
73
product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
NaN                                   610
eletronicos                           517
construcao_ferramentas_construcao     400
Name: count, dtype: int64


#### English translation

In [7]:
category_translation = pd.read_csv(
    RAW_DIR / "product_category_name_translation.csv"
)

In [8]:
display(category_translation.head())

print("Translation rows:", len(category_translation))
print(
    "Unique Portuguese categories:",
    category_translation["product_category_name"].nunique()
)

,product_category_name,product_category_name_english
0,beleza_saude,health_beauty
1,informatica_acessorios,computers_accessories
2,automotivo,auto
3,cama_mesa_banho,bed_bath_table
4,moveis_decoracao,furniture_decor


Translation rows: 71
Unique Portuguese categories: 71


In [9]:
products_translated = products.merge(
    category_translation,
    on="product_category_name",
    how="left"
)

print(
    "Products missing English category translation:",
    products_translated["product_category_name_english"].isna().sum()
)

Products missing English category translation: 623


In [10]:
print(
    products["product_category_name"]
    .value_counts(dropna=False)
    .head(20)
)

product_category_name
cama_mesa_banho                      3029
esporte_lazer                        2867
moveis_decoracao                     2657
beleza_saude                         2444
utilidades_domesticas                2335
automotivo                           1900
informatica_acessorios               1639
brinquedos                           1411
relogios_presentes                   1329
telefonia                            1134
bebes                                 919
perfumaria                            868
papelaria                             849
fashion_bolsas_e_acessorios           849
cool_stuff                            789
ferramentas_jardim                    753
pet_shop                              719
NaN                                   610
eletronicos                           517
construcao_ferramentas_construcao     400
Name: count, dtype: int64


In [11]:
print(
    products_translated["product_category_name_english"]
    .value_counts(dropna=False)
    .head(20)
)

product_category_name_english
bed_bath_table                     3029
sports_leisure                     2867
furniture_decor                    2657
health_beauty                      2444
housewares                         2335
auto                               1900
computers_accessories              1639
toys                               1411
watches_gifts                      1329
telephony                          1134
baby                                919
perfumery                           868
stationery                          849
fashion_bags_accessories            849
cool_stuff                          789
garden_tools                        753
pet_shop                            719
NaN                                 623
electronics                         517
construction_tools_construction     400
Name: count, dtype: int64


##### Investigating categories without english translation

In [14]:
# Products that have a Portuguese category
# but did not receive an English translation
missing_translation = products_translated[
    products_translated["product_category_name"].notna()
    & products_translated["product_category_name_english"].isna()
]

print("Products with category but missing English translation:",
      len(missing_translation))

print("\nUntranslated Portuguese categories:")
print(
    missing_translation["product_category_name"]
    .value_counts()
)

Products with category but missing English translation: 13

Untranslated Portuguese categories:
product_category_name
portateis_cozinha_e_preparadores_de_alimentos    10
pc_gamer                                          3
Name: count, dtype: int64


In [15]:
product_categories = set(
    products["product_category_name"].dropna().unique()
)

translated_categories = set(
    category_translation["product_category_name"].dropna().unique()
)

missing_categories = sorted(
    product_categories - translated_categories
)

print("Categories missing from translation table:")
for category in missing_categories:
    print("-", category)

Categories missing from translation table:
- pc_gamer
- portateis_cozinha_e_preparadores_de_alimentos


In [16]:
print("Number of untranslated category names:", len(missing_categories))

Number of untranslated category names: 2


In [17]:
manual_translations = {
    "pc_gamer": "pc_gamer",
    "portateis_cozinha_e_preparadores_de_alimentos":
        "portable_kitchen_and_food_preparation_appliances"
}

products_translated["product_category_name_english"] = (
    products_translated["product_category_name_english"]
    .fillna(
        products_translated["product_category_name"]
        .map(manual_translations)
    )
)

In [18]:
print(
    "Products still missing English category:",
    products_translated["product_category_name_english"].isna().sum()
)

Products still missing English category: 610


### Check ID fields for missing values

In [12]:
id_checks = {
    "orders.order_id": orders["order_id"],
    "orders.customer_id": orders["customer_id"],
    "items.order_id": items["order_id"],
    "items.product_id": items["product_id"],
    "items.seller_id": items["seller_id"],
    "reviews.order_id": reviews["order_id"],
    "payments.order_id": payments["order_id"],
    "products.product_id": products["product_id"],
    "sellers.seller_id": sellers["seller_id"],
    "customers.customer_id": customers["customer_id"]
}

for name, series in id_checks.items():
    print(
        f"{name}: "
        f"{series.isna().sum()} missing"
    )

orders.order_id: 0 missing
orders.customer_id: 0 missing
items.order_id: 0 missing
items.product_id: 0 missing
items.seller_id: 0 missing
reviews.order_id: 0 missing
payments.order_id: 0 missing
products.product_id: 0 missing
sellers.seller_id: 0 missing
customers.customer_id: 0 missing


### Checking for whitespace and case issues

In [13]:
categorical_columns = {
    "orders.order_status": orders["order_status"],
    "payments.payment_type": payments["payment_type"],
    "customers.customer_city": customers["customer_city"],
    "customers.customer_state": customers["customer_state"],
    "sellers.seller_city": sellers["seller_city"],
    "sellers.seller_state": sellers["seller_state"],
    "products.product_category_name": products["product_category_name"]
}

for name, series in categorical_columns.items():
    stripped = series.dropna().astype(str).str.strip()

    print(name)
    print("Leading/trailing whitespace differences:",
          (series.dropna().astype(str) != stripped).sum())
    print()

orders.order_status
Leading/trailing whitespace differences: 0

payments.payment_type
Leading/trailing whitespace differences: 0

customers.customer_city
Leading/trailing whitespace differences: 0

customers.customer_state
Leading/trailing whitespace differences: 0

sellers.seller_city
Leading/trailing whitespace differences: 0

sellers.seller_state
Leading/trailing whitespace differences: 0

products.product_category_name
Leading/trailing whitespace differences: 0



# Notes


- No missing values were found in the key identifier fields checked.
- Order status, payment type, state, city, and product-category fields had no leading or trailing whitespace issues.
- Order-status and state-code values appeared structurally consistent.
- Seller city values contain some inconsistent labels, so city-level analysis may require additional cleaning later.
- The products table contains 73 non-null Portuguese product categories, while the translation table contains 71 categories.
- Two categories were missing from the translation table: `pc_gamer` and `portateis_cozinha_e_preparadores_de_alimentos`, affecting 13 products.
- Manual English labels were added for those two categories.
- After the manual mappings, 610 products still have no English category because their original product category is missing.
- Original Portuguese category values were retained.
- No identifier values were changed.



---


# Numeric and Product Cleaning




In [19]:
# Main numeric fields to inspect
numeric_checks = {
    "items.price": items["price"],
    "items.freight_value": items["freight_value"],
    "payments.payment_value": payments["payment_value"],
    "payments.payment_installments": payments["payment_installments"],
    "products.product_name_lenght": products["product_name_lenght"],
    "products.product_description_lenght": products["product_description_lenght"],
    "products.product_photos_qty": products["product_photos_qty"],
    "products.product_weight_g": products["product_weight_g"],
    "products.product_length_cm": products["product_length_cm"],
    "products.product_height_cm": products["product_height_cm"],
    "products.product_width_cm": products["product_width_cm"],
}

for name, series in numeric_checks.items():
    print("\n" + name)
    print("Missing:", series.isna().sum())
    print("Min:", series.min())
    print("Max:", series.max())
    print("Zero values:", (series == 0).sum())
    print("Negative values:", (series < 0).sum())


items.price
Missing: 0
Min: 0.85
Max: 6735.0
Zero values: 0
Negative values: 0

items.freight_value
Missing: 0
Min: 0.0
Max: 409.68
Zero values: 383
Negative values: 0

payments.payment_value
Missing: 0
Min: 0.0
Max: 13664.08
Zero values: 9
Negative values: 0

payments.payment_installments
Missing: 0
Min: 0
Max: 24
Zero values: 2
Negative values: 0

products.product_name_lenght
Missing: 610
Min: 5.0
Max: 76.0
Zero values: 0
Negative values: 0

products.product_description_lenght
Missing: 610
Min: 4.0
Max: 3992.0
Zero values: 0
Negative values: 0

products.product_photos_qty
Missing: 610
Min: 1.0
Max: 20.0
Zero values: 0
Negative values: 0

products.product_weight_g
Missing: 2
Min: 0.0
Max: 40425.0
Zero values: 4
Negative values: 0

products.product_length_cm
Missing: 2
Min: 7.0
Max: 105.0
Zero values: 0
Negative values: 0

products.product_height_cm
Missing: 2
Min: 2.0
Max: 105.0
Zero values: 0
Negative values: 0

products.product_width_cm
Missing: 2
Min: 6.0
Max: 118.0
Zero values: 0

In [20]:
product_numeric_cols = [
    "product_name_lenght",
    "product_description_lenght",
    "product_photos_qty",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
]

display(
    products[product_numeric_cols]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
product_name_lenght,32341.0,48.476949,10.245741,5.0,42.0,51.0,57.0,76.0
product_description_lenght,32341.0,771.495285,635.115225,4.0,339.0,595.0,972.0,3992.0
product_photos_qty,32341.0,2.188986,1.736766,1.0,1.0,1.0,3.0,20.0
product_weight_g,32949.0,2276.472488,4282.038731,0.0,300.0,700.0,1900.0,40425.0
product_length_cm,32949.0,30.815078,16.914458,7.0,18.0,25.0,38.0,105.0
product_height_cm,32949.0,16.937661,13.637554,2.0,8.0,13.0,21.0,105.0
product_width_cm,32949.0,23.196728,12.079047,6.0,15.0,20.0,30.0,118.0


In [21]:
print(
    payments["payment_installments"]
    .value_counts()
    .sort_index()
)

payment_installments
0         2
1     52546
2     12413
3     10461
4      7098
5      5239
6      3920
7      1626
8      4268
9       644
10     5328
11       23
12      133
13       16
14       15
15       74
16        5
17        8
18       27
20       17
21        3
22        1
23        1
24       18
Name: count, dtype: int64


In [22]:
missing_product_measurements = products[
    products[
        [
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ].isna().any(axis=1)
]

print("Products missing physical measurements:",
      len(missing_product_measurements))

display(
    missing_product_measurements[
        [
            "product_id",
            "product_category_name",
            "product_weight_g",
            "product_length_cm",
            "product_height_cm",
            "product_width_cm",
        ]
    ]
)

Products missing physical measurements: 2


,product_id,product_category_name,product_weight_g,product_length_cm,product_height_cm,product_width_cm
8578,09ff539a621711667c43eba6a3bd8466,bebes,NaN,NaN,NaN,NaN
18851,5eb564652db742ff8f28759cd8d2652a,NaN,NaN,NaN,NaN,NaN


In [23]:
print("Zero price rows:")
display(items[items["price"] == 0].head(20))

print("Zero freight rows:")
display(items[items["freight_value"] == 0].head(20))

print("Zero payment value rows:")
display(payments[payments["payment_value"] == 0].head(20))

print("Zero installment rows:")
display(payments[payments["payment_installments"] == 0].head(20))

Zero price rows:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value


Zero freight rows:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
114,00404fa7a687c8c44ca69d42695aae73,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-15 04:31:26,99.9,0.0
258,00a870c6c06346e85335524935c600c0,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-14 00:14:29,69.9,0.0
483,011c899816ea29773525bd3322dbb6aa,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-07 05:30:45,99.9,0.0
508,012b3f6ab7776a8ab3443a4ad7bef2e6,1,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
509,012b3f6ab7776a8ab3443a4ad7bef2e6,2,422879e10f46682990de24d770e7f83d,1f50f920176fa81dab994f9023523100,2018-05-09 21:30:50,53.9,0.0
1784,04105b54650921ca3246f52e6f175f46,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-04-27 09:31:35,69.9,0.0
2232,0517a3e68dac3308995edca2144db36e,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-03 19:11:40,99.9,0.0
2714,061ba2e2d7544790b6ed6b5b4dd9278c,1,53b36df67ebb7c41585e8d54d6772e08,7d13fca15225358621be4086e1eb0964,2018-05-09 23:10:46,110.0,0.0
3224,07441f525824bc6b31d4dc19c5d49fc9,1,53b36df67ebb7c41585e8d54d6772e08,4869f7a5dfa277a7dca6462dcf3b52b2,2018-04-26 13:31:30,106.9,0.0
3378,079f16689c29acb6cab92978e6af2137,1,aca2eb7d00ea1a7b8ebd4e68314663af,955fee9216a65b617aa5c0531780ce60,2018-05-11 18:15:13,69.9,0.0


Zero payment value rows:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
19922,8bcbe01d44d147f901cd3192671144db,4,voucher,1,0.0
36822,fa65dad1b0e818e3ccc5cb0e39231352,14,voucher,1,0.0
43744,6ccb433e00daae1283ccc956189c82ae,4,voucher,1,0.0
51280,4637ca194b6387e2d538dc89b124b0ee,1,not_defined,1,0.0
57411,00b1cb0320190ca0daa2c88b35206009,1,not_defined,1,0.0
62674,45ed6e85398a87c253db47c2d9f48216,3,voucher,1,0.0
77885,fa65dad1b0e818e3ccc5cb0e39231352,13,voucher,1,0.0
94427,c8c528189310eaa44a745b8d9d26908b,1,not_defined,1,0.0
100766,b23878b3e8eb4d25a158f57d96331b18,4,voucher,1,0.0


Zero installment rows:


,order_id,payment_sequential,payment_type,payment_installments,payment_value
46982,744bade1fcf9ff3f31d860ace076d422,2,credit_card,0,58.69
79014,1a57108394169c0b47d8f876acc9ba2d,2,credit_card,0,129.94


# Notes

- No negative values were found in the numeric fields checked.
- Item prices were all positive, with no zero-price records.
- 383 order-item records have zero freight value; these may represent free shipping and will be retained for now.
- 9 payment records have a payment value of 0, mostly involving vouchers or `not_defined` payment types; these will be flagged rather than removed.
- 2 credit-card payment records have 0 installments, which appears unusual and will require a later decision.
- Product name length, description length, and photo count are missing for 610 products.
- Only 2 products are missing weight and dimension measurements.
- Product weight includes 4 zero values, which will be reviewed before using weight in analysis.
- No numeric records were removed or modified during this step.



---


# Date Cleaning


### Converting the date columns

In [24]:
# Orders
order_date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in order_date_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

# Order items
items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"],
    errors="coerce"
)

# Reviews
reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"],
    errors="coerce"
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"],
    errors="coerce"
)

### Verifying types

In [25]:
print(orders[order_date_cols].dtypes)

print("\nOrder items:")
print(items[["shipping_limit_date"]].dtypes)

print("\nReviews:")
print(
    reviews[
        ["review_creation_date", "review_answer_timestamp"]
    ].dtypes
)

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Order items:
shipping_limit_date    datetime64[ns]
dtype: object

Reviews:
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


### Check missing values after conversion

In [26]:
print("Orders:")
print(orders[order_date_cols].isna().sum())

print("\nOrder items:")
print(items[["shipping_limit_date"]].isna().sum())

print("\nReviews:")
print(
    reviews[
        ["review_creation_date", "review_answer_timestamp"]
    ].isna().sum()
)

Orders:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Order items:
shipping_limit_date    0
dtype: int64

Reviews:
review_creation_date       0
review_answer_timestamp    0
dtype: int64


### Checking date logic

In [27]:
print(
    "Approval before purchase:",
    (
        orders["order_approved_at"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Carrier date before purchase:",
    (
        orders["order_delivered_carrier_date"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before purchase:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_purchase_timestamp"]
    ).sum()
)

print(
    "Customer delivery before carrier delivery:",
    (
        orders["order_delivered_customer_date"]
        < orders["order_delivered_carrier_date"]
    ).sum()
)

print(
    "Review answer before review creation:",
    (
        reviews["review_answer_timestamp"]
        < reviews["review_creation_date"]
    ).sum()
)

Approval before purchase: 0
Carrier date before purchase: 166
Customer delivery before purchase: 0
Customer delivery before carrier delivery: 23
Review answer before review creation: 0


### Inspecting date ranges

In [28]:
date_ranges = {
    "Purchase": orders["order_purchase_timestamp"],
    "Approval": orders["order_approved_at"],
    "Carrier": orders["order_delivered_carrier_date"],
    "Customer delivery": orders["order_delivered_customer_date"],
    "Estimated delivery": orders["order_estimated_delivery_date"],
    "Shipping limit": items["shipping_limit_date"],
    "Review creation": reviews["review_creation_date"],
    "Review answer": reviews["review_answer_timestamp"],
}

for name, series in date_ranges.items():
    print(
        f"{name}: "
        f"{series.min()} -> {series.max()}"
    )

Purchase: 2016-09-04 21:15:19 -> 2018-10-17 17:30:18
Approval: 2016-09-15 12:16:38 -> 2018-09-03 17:40:06
Carrier: 2016-10-08 10:34:01 -> 2018-09-11 19:48:28
Customer delivery: 2016-10-11 13:46:32 -> 2018-10-17 13:22:46
Estimated delivery: 2016-09-30 00:00:00 -> 2018-11-12 00:00:00
Shipping limit: 2016-09-19 00:15:34 -> 2020-04-09 22:35:08
Review creation: 2016-10-02 00:00:00 -> 2018-08-31 00:00:00
Review answer: 2016-10-07 18:32:28 -> 2018-10-29 12:27:35


### Looking at concerning outputs

In [29]:
# Carrier date before purchase
carrier_before_purchase = orders[
    orders["order_delivered_carrier_date"]
    < orders["order_purchase_timestamp"]
]

print("Carrier before purchase:", len(carrier_before_purchase))
display(
    carrier_before_purchase[
        [
            "order_id",
            "order_status",
            "order_purchase_timestamp",
            "order_delivered_carrier_date"
        ]
    ].head(20)
)

Carrier before purchase: 166


,order_id,order_status,order_purchase_timestamp,order_delivered_carrier_date
615,b9afddbdcfadc9a87b41a83271c3e888,delivered,2018-08-16 13:50:48,2018-08-16 13:27:00
1111,ad133696906f6a78826daa0911b7daec,delivered,2018-06-15 15:41:22,2018-06-15 14:52:00
1329,74e033208dc13a7b8127eb8e73d09b76,delivered,2018-05-02 10:48:44,2018-05-02 09:49:00
1372,a6b58794fd2ba533359a76c08df576e3,delivered,2018-05-14 15:18:23,2018-05-14 13:46:00
1864,5792e0b1c8c8a2bf53af468c9a422c88,delivered,2018-07-26 13:25:14,2018-07-26 12:42:00
2760,c3eb293fd154223498b6551a728203e8,delivered,2018-07-19 14:06:04,2018-07-19 13:49:00
3473,b0c2a7d04b165525254254a728c50a4e,delivered,2018-06-07 13:28:30,2018-06-07 13:22:00
3661,2033a4586b5bec3229ebc1675a8ae092,delivered,2018-06-12 10:10:25,2018-06-12 10:09:00
4114,08adcddad19d3acf37d1fa01cb9ded1e,delivered,2018-06-27 11:16:44,2018-06-27 10:57:00
4159,dee6298ce7d1fb2645141ef9972157aa,shipped,2018-04-30 14:06:12,2018-04-30 12:59:00


In [30]:
# Customer delivery before carrier date
delivery_before_carrier = orders[
    orders["order_delivered_customer_date"]
    < orders["order_delivered_carrier_date"]
]

print("Delivery before carrier:", len(delivery_before_carrier))
display(
    delivery_before_carrier[
        [
            "order_id",
            "order_status",
            "order_delivered_carrier_date",
            "order_delivered_customer_date"
        ]
    ].head(20)
)

Delivery before carrier: 23


,order_id,order_status,order_delivered_carrier_date,order_delivered_customer_date
6437,a1abeb653a4d4cd1e142ccb8c82cd069,delivered,2017-07-28 16:57:58,2017-07-25 19:32:56
9553,383aa8b2724fe452d9ccd9934a8c628b,delivered,2017-07-07 17:22:41,2017-07-06 14:27:51
13487,cb1134f9010d242e9515ad1c78ec0c39,delivered,2017-07-20 19:22:02,2017-07-19 14:13:28
14474,dceb62e8fa94b46006c9554fed743df0,delivered,2017-08-01 18:23:30,2017-07-26 18:09:10
19268,5f9d46795c3126674e52becb3a1a517f,delivered,2017-07-20 23:03:42,2017-07-20 18:52:41
21338,8c78d01de3a9009e23d6877a7cc9be20,delivered,2016-10-26 11:41:53,2016-10-25 17:51:46
22520,b27af682321527a6349f1761eb3f360c,delivered,2017-06-27 14:51:54,2017-06-26 15:45:35
25393,1cc3ae63caffff2d6c3ee3e78e074acf,delivered,2017-08-10 18:28:56,2017-08-10 18:05:38
25646,e37f11cae9985ca58f0b56f268720537,delivered,2017-08-01 18:17:47,2017-07-31 17:49:56
27470,fa3e37584f4fdb1ded0e0de700dfcb4e,delivered,2017-08-09 18:18:43,2017-08-01 21:13:01


In [31]:
# Suspicious shipping-limit dates after the main dataset period
late_shipping_limits = items[
    items["shipping_limit_date"] > pd.Timestamp("2018-12-31")
]

print("Shipping-limit dates after 2018:", len(late_shipping_limits))

display(
    late_shipping_limits[
        ["order_id", "seller_id", "shipping_limit_date"]
    ].sort_values("shipping_limit_date", ascending=False)
)

Shipping-limit dates after 2018: 4


,order_id,seller_id,shipping_limit_date
85729,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08
85730,c2bb89b5c1dd978d507284be78a04cb2,7a241947449cc45dbfda4f9d0798d9d0,2020-04-09 22:35:08
8643,13bdf405f961a6deec817d817f5c6624,7a241947449cc45dbfda4f9d0798d9d0,2020-02-05 03:30:51
68516,9c94a4ea2f7876660fa6f1b59b69c8e6,7a241947449cc45dbfda4f9d0798d9d0,2020-02-03 20:23:22


# Notes

- All intended date/time fields were successfully converted to datetime format.
- Missing date values remained consistent with the original data; conversion did not introduce unexpected missing values.
- No approvals occurred before purchase, no customer deliveries occurred before purchase, and no review responses occurred before review creation.
- 166 orders have a carrier-delivery timestamp earlier than the purchase timestamp.
- 23 delivered orders have a customer-delivery timestamp earlier than the carrier timestamp.
- These inconsistent date sequences will be treated as invalid for duration calculations rather than manually corrected.
- Four order-item records have shipping-limit dates in 2020, outside the main dataset period; these involve 3 orders from the same seller and will be treated as anomalous if that field is used.
- No raw date values were overwritten.



---


# Feature Engineering v1

### Creating the order-level delivery variables

In [37]:
# Delivery time in days
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.total_seconds() / 86400

# Difference between actual and estimated delivery
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.total_seconds() / 86400

# Late delivery flag
orders["late_delivery_flag"] = pd.Series(
    pd.NA,
    index=orders.index,
    dtype="Int64"
)

orders.loc[
    orders["delivery_delay_days"].notna(),
    "late_delivery_flag"
] = (
    orders.loc[
        orders["delivery_delay_days"].notna(),
        "delivery_delay_days"
    ] > 0
).astype("Int64")

### Nulling out impossible delivery durations

In [38]:
orders.loc[
    orders["delivery_days"] < 0,
    "delivery_days"
] = pd.NA

### Order-level item totals

In [39]:
order_item_summary = (
    items
    .groupby("order_id", as_index=False)
    .agg(
        order_value=("price", "sum"),
        freight_total=("freight_value", "sum"),
        items_per_order=("order_item_id", "count")
    )
)

order_item_summary["freight_percentage"] = (
    order_item_summary["freight_total"]
    / order_item_summary["order_value"]
    * 100
)

### Inspecting new variables

In [40]:
display(
    orders[
        [
            "delivery_days",
            "delivery_delay_days",
            "late_delivery_flag"
        ]
    ].describe()
)

display(
    order_item_summary[
        [
            "order_value",
            "freight_total",
            "freight_percentage",
            "items_per_order"
        ]
    ].describe()
)

,delivery_days,delivery_delay_days,late_delivery_flag
count,96476.000000,96476.000000,96476.0
mean,12.558702,-11.179120,0.081129
std,9.546530,10.186113,0.273035
min,0.533414,-146.016123,0.0
25%,6.766403,-16.244384,0.0
50%,10.217755,-11.948941,0.0
75%,15.720327,-6.390000,0.0
max,209.628611,188.975081,1.0


,order_value,freight_total,freight_percentage,items_per_order
count,98666.000000,98666.000000,98666.000000,98666.000000
mean,137.754076,22.823562,30.838869,1.141731
std,210.645145,21.650909,31.476219,0.538452
min,0.850000,0.000000,0.000000,1.000000
25%,45.900000,13.850000,13.186441,1.000000
50%,86.900000,17.170000,22.437396,1.000000
75%,149.900000,24.040000,38.019087,1.000000
max,13440.000000,1794.960000,2144.705882,21.000000


### Sanity checks

In [41]:
print(
    "Missing delivery_days:",
    orders["delivery_days"].isna().sum()
)

print(
    "Late deliveries:",
    orders["late_delivery_flag"].sum()
)

print(
    "Orders in item summary:",
    len(order_item_summary)
)

print(
    "Zero or negative order values:",
    (order_item_summary["order_value"] <= 0).sum()
)

print(
    "Missing freight percentages:",
    order_item_summary["freight_percentage"].isna().sum()
)

Missing delivery_days: 2965
Late deliveries: 7827
Orders in item summary: 98666
Zero or negative order values: 0
Missing freight percentages: 0


# Notes

- Created `delivery_days`, `delivery_delay_days`, and `late_delivery_flag`.
- Orders without an actual delivery date remain missing for delivery-based measures rather than being classified as not late.
- 96,476 orders have valid delivery-based measures, and 7,827 were delivered late.
- Created order-level `order_value`, `freight_total`, `freight_percentage`, and `items_per_order`.
- 98,666 orders had item records available for the order-level item summary.
- No zero or negative order values and no missing freight percentages were found.
- Extreme delivery times and freight percentages remain in the data for later outlier/EDA review rather than being removed automatically.